# 06 สรุปข้อเสนอเชิงแนวทางและผลการวิเคราะห์

Notebook นี้รวบรวมหลักฐานจาก notebook 01–05 ได้แก่ การค้นหาชุดข้อมูล การตรวจ schema การวิเคราะห์เชิงพรรณนาและเชิงวินิจฉัย รายงาน interactive และการทดลอง ML เบื้องต้น
ผลลัพธ์ใช้เพื่อจัดทำข้อเสนอเรื่องข้อมูลและแนวทางวิเคราะห์ต่อไป ไม่สร้าง model ใหม่ และไม่ใช้จัดอันดับ ตัดสิน หรือกำหนดสถานะของจังหวัด


## 01 นำเข้าไลบรารี

ใช้เฉพาะไลบรารีมาตรฐานและ pandas เพื่ออ่าน cleaned output สรุปตัวชี้วัดที่สังเกตได้ และสร้างรายงาน HTML แบบ static

In [1]:
from pathlib import Path

import pandas as pd

## 02 ค้นหา root ของโครงการ

รองรับการเปิด notebook จาก `notebooks/` หรือจาก root ของโครงการ โดยตรวจหา `data/raw/` ที่มีอยู่จริง

In [2]:
def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path]
    candidate_paths.extend(current_path.parents)

    for candidate_path in candidate_paths:
        if (candidate_path / "data" / "raw").exists():
            return candidate_path

    raise FileNotFoundError("ไม่พบ root ของโครงการที่มี data/raw")


project_root = find_project_root()

## 03 กำหนดตำแหน่งไฟล์

ระบุ path ของ cleaned data ที่ใช้สรุปตัวชี้วัดและสร้างรายงาน


In [3]:
processed_data_path = project_root / "data" / "processed" / "labai_demand_supply_cleaned.csv"
final_report_path = project_root / "reports" / "html" / "final_summary.html"


## 04 ตรวจสอบไฟล์ข้อมูล

ยืนยันว่า cleaned data มีอยู่จริงก่อนอ่าน


In [4]:
if not processed_data_path.exists():
    raise FileNotFoundError(f"ไม่พบไฟล์ข้อมูล: {processed_data_path}")

data_file_status = pd.DataFrame(
    {
        "ไฟล์": [str(processed_data_path.relative_to(project_root))],
        "มีอยู่": [processed_data_path.exists()],
    }
)

data_file_status


,ไฟล์,มีอยู่
0,data\processed\labai_demand_supply_cleaned.csv,True


## 05 อ่านข้อมูล cleaned

ใช้ไฟล์ demand/supply ที่ผ่าน cleaning แล้ว และคงขอบเขตเป็นข้อมูล 77 แถวระดับจังหวัดที่ดาวน์โหลดในรอบโครงการนี้


In [5]:
demand_supply = pd.read_csv(processed_data_path)
demand_supply.head()

,PROVINCE_NAME,countw,count_want_land,area_rai_want_land,area_nga_want_land,area_wa_want_land,countowner,count_owner_land,area_rai_owner,area_nga_owner,...,province_name,area_rai_want_land_numeric,area_rai_want_land_has_placeholder,area_nga_want_land_numeric,area_nga_want_land_has_placeholder,area_rai_owner_numeric,area_rai_owner_has_placeholder,area_nga_owner_numeric,area_nga_owner_has_placeholder,count_gap_want_minus_owner
0,กรุงเทพมหานคร,164,39,0,2,53.0,36,1,0,2,...,กรุงเทพมหานคร,0,False,2,False,0,False,2,False,38
1,สมุทรปราการ,29,22,0,0,0.0,6,0,0,0,...,สมุทรปราการ,0,False,0,False,0,False,0,False,22
2,นนทบุรี,35,30,0,0,0.0,6,0,0,0,...,นนทบุรี,0,False,0,False,0,False,0,False,30
3,ปทุมธานี,44,72,1,0,0.0,2,1,1,0,...,ปทุมธานี,1,False,0,False,1,False,0,False,71
4,พระนครศรีอยุธยา,15,19,0,0,0.0,3,0,0,0,...,พระนครศรีอยุธยา,0,False,0,False,0,False,0,False,19


## 06 ตรวจ column ที่ใช้สรุป

ใช้เฉพาะ column ที่ตรวจพบจริงใน cleaned output เพื่อป้องกันการสร้างความหมายหรือ schema เพิ่มเอง

In [6]:
summary_columns = [
    "province_name",
    "count_want_land",
    "count_owner_land",
    "count_gap_want_minus_owner",
]
missing_summary_columns = [column for column in summary_columns if column not in demand_supply.columns]

if missing_summary_columns:
    raise KeyError(f"ไม่พบ column ที่ต้องใช้: {missing_summary_columns}")

available_summary_columns = demand_supply[summary_columns].dtypes.reset_index()
available_summary_columns.columns = ["column", "data_type"]
available_summary_columns

,column,data_type
0,province_name,object
1,count_want_land,int64
2,count_owner_land,int64
3,count_gap_want_minus_owner,int64


## 07 คำนวณผลรวมที่สังเกตได้

ผลรวมนี้อธิบายเฉพาะข้อมูล cleaned 77 แถว ไม่ใช่จำนวนสะสมทุกช่วงเวลาหรือผลยืนยันระดับหน่วยงาน

In [7]:
total_want_land = int(demand_supply["count_want_land"].sum())
total_owner_land = int(demand_supply["count_owner_land"].sum())
row_count = int(demand_supply.shape[0])

observed_total_metrics = pd.DataFrame(
    {
        "ตัวชี้วัด": ["จำนวนแถว", "ผลรวมผู้ประสงค์ใช้ที่ดิน", "ผลรวมเจ้าของที่ดิน"],
        "ค่า": [row_count, total_want_land, total_owner_land],
    }
)
observed_total_metrics

,ตัวชี้วัด,ค่า
0,จำนวนแถว,77
1,ผลรวมผู้ประสงค์ใช้ที่ดิน,1371
2,ผลรวมเจ้าของที่ดิน,80


## 08 สรุปสัญญาณของส่วนต่าง

ส่วนต่างคำนวณจาก `count_want_land - count_owner_land` เพื่อการตรวจข้อมูลต่อ ไม่ใช่ตัวชี้วัดสถานะหรือคำตัดสินของจังหวัด

In [8]:
positive_gap_count = int((demand_supply["count_gap_want_minus_owner"] > 0).sum())
negative_gap_count = int((demand_supply["count_gap_want_minus_owner"] < 0).sum())
zero_gap_count = int((demand_supply["count_gap_want_minus_owner"] == 0).sum())

observed_gap_metrics = pd.DataFrame(
    {
        "กลุ่มส่วนต่างเชิงคำนวณ": ["มากกว่าศูนย์", "น้อยกว่าศูนย์", "เท่ากับศูนย์"],
        "จำนวนแถว": [positive_gap_count, negative_gap_count, zero_gap_count],
    }
)
observed_gap_metrics

,กลุ่มส่วนต่างเชิงคำนวณ,จำนวนแถว
0,มากกว่าศูนย์,68
1,น้อยกว่าศูนย์,3
2,เท่ากับศูนย์,6


## 09 คำนวณความสัมพันธ์ของจำนวนผู้ประสงค์และเจ้าของที่ดิน

Pearson correlation ใช้ดูความสัมพันธ์เชิงเส้นในไฟล์นี้เท่านั้น ไม่ใช้ยืนยันสาเหตุหรือประสิทธิภาพการดำเนินงาน

In [9]:
demand_supply_correlation = demand_supply["count_want_land"].corr(
    demand_supply["count_owner_land"]
)

observed_relationship_metric = pd.DataFrame(
    {
        "ตัวชี้วัด": ["Pearson correlation ระหว่างจำนวนผู้ประสงค์และเจ้าของที่ดิน"],
        "ค่า": [round(float(demand_supply_correlation), 3)],
    }
)
observed_relationship_metric

,ตัวชี้วัด,ค่า
0,Pearson correlation ระหว่างจำนวนผู้ประสงค์และเ...,0.466


## 10 สรุปแหล่งข้อมูล

URL ต่อไปนี้เป็นหน้า dataset ทางการที่บันทึกไว้ในเอกสารของโครงการ การตีความตัวแปรยังต้องอาศัย metadata และการยืนยันจากหน่วยงานเมื่อมีการใช้งานจริง

In [10]:
dataset_sources = pd.DataFrame(
    {
        "ชุดข้อมูล": [
            "จำนวนผู้ประสงค์ใช้ที่ดินและเจ้าของที่ดินที่ลงทะเบียน",
            "ข้อมูลการจับคู่ที่ดิน",
            "จำนวนเจ้าของที่ดินที่ลงทะเบียน",
        ],
        "แหล่งข้อมูลทางการ": [
            "https://data.go.th/dataset/65_dataset_11_03",
            "https://data.go.th/dataset/65_dataset_21_01",
            "https://data.go.th/dataset/65_dataset_11_01",
        ],
        "บทบาทในโครงการ": [
            "ข้อมูลหลักสำหรับการวิเคราะห์และการทดลองโมเดล baseline",
            "ใช้ตรวจข้อจำกัดของค่า placeholder",
            "ใช้ตรวจความสอดคล้องของจำนวนเจ้าของที่ดิน",
        ],
    }
)
dataset_sources

,ชุดข้อมูล,แหล่งข้อมูลทางการ,บทบาทในโครงการ
0,จำนวนผู้ประสงค์ใช้ที่ดินและเจ้าของที่ดินที่ลงท...,https://data.go.th/dataset/65_dataset_11_03,ข้อมูลหลักสำหรับการวิเคราะห์และการทดลองโมเดล b...
1,ข้อมูลการจับคู่ที่ดิน,https://data.go.th/dataset/65_dataset_21_01,ใช้ตรวจข้อจำกัดของค่า placeholder
2,จำนวนเจ้าของที่ดินที่ลงทะเบียน,https://data.go.th/dataset/65_dataset_11_01,ใช้ตรวจความสอดคล้องของจำนวนเจ้าของที่ดิน


## 11 เส้นทางจากข้อมูลดิบสู่ข้อเสนอ

ตารางนี้แสดงขั้นตอนที่ดำเนินการในโครงการ ตั้งแต่การค้นหาข้อมูลไปจนถึงการจัดทำข้อเสนอ


In [11]:
workflow_steps = pd.DataFrame(
    {
        "ขั้นตอน": [
            "ค้นหาและคัดเลือกชุดข้อมูล",
            "ดาวน์โหลดและตรวจโครงสร้างข้อมูล",
            "ทำความสะอาดและวิเคราะห์ข้อมูล",
            "สร้างรายงานอ่านง่ายและกราฟ interactive",
            "ทดลอง ML เบื้องต้น",
            "สรุปข้อเสนอและข้อจำกัด",
        ],
        "รายละเอียด": [
            "ค้นหาและลงทะเบียน candidate datasets ของ LABAI, SACIT และ KPI พร้อม URL ทางการ",
            "ดาวน์โหลดไฟล์ที่เข้าถึงได้และตรวจ schema จริงจากไฟล์ที่อ่านได้",
            "ทำ cleaning, วิเคราะห์เชิงพรรณนาและวินิจฉัย และสร้าง processed CSV",
            "สร้างรายงาน interactive ด้วย Plotly สำหรับดูค่ารายจังหวัด",
            "ทดลอง clustering และ classification baseline ภายใต้ข้อจำกัดของข้อมูล",
            "สรุปข้อเสนอเชิงแนวทาง, ข้อจำกัด และจัดทำ notebook นี้",
        ],
        "สถานะ": ["เสร็จสิ้น", "เสร็จสิ้น", "เสร็จสิ้น", "เสร็จสิ้น", "เสร็จสิ้น", "เสร็จสิ้น"],
    }
)
workflow_steps


,ขั้นตอน,รายละเอียด,สถานะ
0,ค้นหาและคัดเลือกชุดข้อมูล,ค้นหาและลงทะเบียน candidate datasets ของ LABAI...,เสร็จสิ้น
1,ดาวน์โหลดและตรวจโครงสร้างข้อมูล,ดาวน์โหลดไฟล์ที่เข้าถึงได้และตรวจ schema จริงจ...,เสร็จสิ้น
2,ทำความสะอาดและวิเคราะห์ข้อมูล,"ทำ cleaning, วิเคราะห์เชิงพรรณนาและวินิจฉัย แล...",เสร็จสิ้น
3,สร้างรายงานอ่านง่ายและกราฟ interactive,สร้างรายงาน interactive ด้วย Plotly สำหรับดูค่...,เสร็จสิ้น
4,ทดลอง ML เบื้องต้น,ทดลอง clustering และ classification baseline ภ...,เสร็จสิ้น
5,สรุปข้อเสนอและข้อจำกัด,"สรุปข้อเสนอเชิงแนวทาง, ข้อจำกัด และจัดทำ noteb...",เสร็จสิ้น


## 12 สรุปการทดลอง ML เบื้องต้น

ตารางนี้นำค่าที่บันทึกไว้จากการรัน notebook `05_labai_predictive_baseline.ipynb` มาอ้างอิง ไม่ได้ฝึกหรือประเมิน model ซ้ำใน notebook นี้


In [12]:
ml_baseline_summary = pd.DataFrame(
    {
        "งาน": ["Clustering", "Classification", "Regression"],
        "วิธีที่บันทึกไว้": [
            "KMeans หลัง StandardScaler, เลือก k=3 เพื่อสาธิตการอ่าน profile",
            "Logistic Regression ของ derived label high_gap",
            "ไม่ดำเนินการ",
        ],
        "หลักฐานที่บันทึกไว้": [
            "silhouette ของ k=3 เท่ากับ 0.6751, ขนาดกลุ่ม 15, 60 และ 2 แถว",
            "split เดียว 20 แถว: accuracy 0.850, precision 0.625, recall 1.000, F1 0.769",
            "ข้ามเพราะความเหมาะสมของ feature กับ target และความเสี่ยง leakage ยังไม่ยืนยัน",
        ],
        "ข้อควรตีความ": [
            "ใช้เพื่อเลือกประเด็นตรวจ metadata และคุณภาพข้อมูลเพิ่ม ไม่ใช่การจัดระดับจังหวัด",
            "label ไม่ใช่สถานะทางการ และ metric ไม่ใช่ผลยืนยันการใช้งานจริง",
            "ไม่ควรตีความว่าไม่มีความเป็นไปได้ในอนาคต",
        ],
    }
)
ml_baseline_summary

,งาน,วิธีที่บันทึกไว้,หลักฐานที่บันทึกไว้,ข้อควรตีความ
0,Clustering,"KMeans หลัง StandardScaler, เลือก k=3 เพื่อสาธ...","silhouette ของ k=3 เท่ากับ 0.6751, ขนาดกลุ่ม 1...",ใช้เพื่อเลือกประเด็นตรวจ metadata และคุณภาพข้อ...
1,Classification,Logistic Regression ของ derived label high_gap,"split เดียว 20 แถว: accuracy 0.850, precision ...",label ไม่ใช่สถานะทางการ และ metric ไม่ใช่ผลยืน...
2,Regression,ไม่ดำเนินการ,ข้ามเพราะความเหมาะสมของ feature กับ target และ...,ไม่ควรตีความว่าไม่มีความเป็นไปได้ในอนาคต


## 13 ข้อเสนอจากผลวิเคราะห์

ข้อเสนออิงจากข้อจำกัดที่ตรวจพบจริงในชุดข้อมูลและจากการทดลอง ML เบื้องต้น โดยมุ่งเพิ่มความพร้อมของข้อมูลก่อนใช้ผลวิเคราะห์ในบริบทที่มีผลกระทบจริง


In [13]:
prescriptive_recommendations = pd.DataFrame(
    {
        "หมวด": [
            "ธรรมาภิบาลข้อมูล",
            "คุณภาพข้อมูล",
            "การวิเคราะห์",
            "ความพร้อมสำหรับ ML",
            "การต่อยอดชุดข้อมูล",
        ],
        "หลักฐานจากโครงการ": [
            "นิยามของ count, area และช่วงเวลายังยืนยันไม่ได้ครบจาก metadata ที่ใช้",
            "ข้อมูลการจับคู่มีค่า placeholder '-' ใน 70 จาก 77 แถว",
            "ข้อมูลปัจจุบันไม่มี column เวลา และส่วนต่างเป็นผลคำนวณจากจำนวนสอง field",
            "มีเพียง 77 แถวและพบ outlier หลาย field ในขั้นตอนการทดลองโมเดล baseline",
            "มีแนวคิด LABAI, SACIT และ KPI ที่ยังไม่ได้ดำเนินการในรายงานนี้",
        ],
        "ข้อเสนอ": [
            "จัดทำ data dictionary, owner ของข้อมูล, วันที่อ้างอิง และกติกาการปรับปรุงข้อมูลให้ตรวจย้อนกลับได้",
            "กำหนดรหัส missing หรือสถานะของ '-' อย่างเป็นทางการ แล้วเก็บค่าและเหตุผลแยกจากศูนย์",
            "เก็บ snapshot ตามเวลาและตรวจหน่วยพื้นที่ก่อนใช้ทำ trend, rate หรือ forecasting",
            "ใช้ cluster เป็นสัญญาณเลือกประเด็นตรวจเพิ่ม, สร้าง label จากนิยามที่อนุมัติ และเพิ่มตัวแปรอธิบายก่อนทำ model ใหม่",
            "ตรวจ schema และเงื่อนไขการเข้าถึงของ SACIT และ KPI ก่อนต่อยอดเป็นแผนที่ แนวโน้ม หรือข้อมูลเชิงข้อความ",
        ],
    }
)
prescriptive_recommendations

,หมวด,หลักฐานจากโครงการ,ข้อเสนอ
0,ธรรมาภิบาลข้อมูล,"นิยามของ count, area และช่วงเวลายังยืนยันไม่ได...","จัดทำ data dictionary, owner ของข้อมูล, วันที่..."
1,คุณภาพข้อมูล,ข้อมูลการจับคู่มีค่า placeholder '-' ใน 70 จาก...,กำหนดรหัส missing หรือสถานะของ '-' อย่างเป็นทา...
2,การวิเคราะห์,ข้อมูลปัจจุบันไม่มี column เวลา และส่วนต่างเป็...,เก็บ snapshot ตามเวลาและตรวจหน่วยพื้นที่ก่อนใช...
3,ความพร้อมสำหรับ ML,มีเพียง 77 แถวและพบ outlier หลาย field ในขั้นต...,"ใช้ cluster เป็นสัญญาณเลือกประเด็นตรวจเพิ่ม, ส..."
4,การต่อยอดชุดข้อมูล,"มีแนวคิด LABAI, SACIT และ KPI ที่ยังไม่ได้ดำเน...",ตรวจ schema และเงื่อนไขการเข้าถึงของ SACIT และ...


## 14 ข้อจำกัดที่ต้องอ่านร่วมกับผลวิเคราะห์

ข้อจำกัดถูกแสดงร่วมกับข้อเสนอเพื่อป้องกันการใช้ตัวเลขหรือผลการทดลองเกินขอบเขตของข้อมูลที่ดาวน์โหลด


In [14]:
final_limitations = pd.DataFrame(
    {
        "ข้อจำกัด": [
            "ข้อมูลหลักมี 77 แถวระดับจังหวัด",
            "ไม่มี column เวลาในไฟล์หลัก",
            "ค่า '-' ในข้อมูลการจับคู่ยังไม่ยืนยันความหมาย",
            "หน่วยและวิธีรวม area fields ยังไม่ยืนยัน",
            "high_gap เป็น label ที่สร้างจาก quantile ไม่ใช่สถานะทางการ",
            "clustering เป็นการสำรวจ pattern เบื้องต้น",
            "classification เป็น baseline เพื่อการเรียนรู้จาก split เดียว",
            "ข้าม regression เพราะ feature-target suitability และความเสี่ยง leakage",
        ],
        "ผลต่อการใช้งาน": [
            "ไม่ควรใช้เปรียบเทียบหรือตัดสินเชิงนโยบาย",
            "ไม่สามารถทำ forecasting ที่มีความหมายจากไฟล์นี้เพียงชุดเดียว",
            "ไม่ควรคำนวณอัตราการจับคู่หรือใช้เป็น target",
            "ไม่ควรรวมหรือเปรียบเทียบพื้นที่จนกว่าจะยืนยัน metadata",
            "ไม่ควรใช้เป็นผลลัพธ์ทางธุรการ",
            "ไม่ใช่การแบ่งระดับความเสี่ยงหรือศักยภาพ",
            "ไม่ใช่ค่าประสิทธิภาพที่ยืนยันการใช้งานจริง",
            "ไม่ควรสรุปว่าการพยากรณ์เหมาะสมหรือไม่เหมาะสมอย่างเด็ดขาด",
        ],
    }
)
final_limitations

,ข้อจำกัด,ผลต่อการใช้งาน
0,ข้อมูลหลักมี 77 แถวระดับจังหวัด,ไม่ควรใช้เปรียบเทียบหรือตัดสินเชิงนโยบาย
1,ไม่มี column เวลาในไฟล์หลัก,ไม่สามารถทำ forecasting ที่มีความหมายจากไฟล์นี...
2,ค่า '-' ในข้อมูลการจับคู่ยังไม่ยืนยันความหมาย,ไม่ควรคำนวณอัตราการจับคู่หรือใช้เป็น target
3,หน่วยและวิธีรวม area fields ยังไม่ยืนยัน,ไม่ควรรวมหรือเปรียบเทียบพื้นที่จนกว่าจะยืนยัน ...
4,high_gap เป็น label ที่สร้างจาก quantile ไม่ใช...,ไม่ควรใช้เป็นผลลัพธ์ทางธุรการ
5,clustering เป็นการสำรวจ pattern เบื้องต้น,ไม่ใช่การแบ่งระดับความเสี่ยงหรือศักยภาพ
6,classification เป็น baseline เพื่อการเรียนรู้จ...,ไม่ใช่ค่าประสิทธิภาพที่ยืนยันการใช้งานจริง
7,ข้าม regression เพราะ feature-target suitabili...,ไม่ควรสรุปว่าการพยากรณ์เหมาะสมหรือไม่เหมาะสมอย...


## 15 เตรียมตาราง HTML

แปลงตารางสรุปเป็น HTML โดยคงข้อความและตัวเลขตามผลที่ตรวจได้จากข้อมูล


In [15]:
dataset_sources_html = dataset_sources.to_html(index=False, escape=True, classes="report-table")
workflow_steps_html = workflow_steps.to_html(index=False, escape=True, classes="report-table")
observed_total_metrics_html = observed_total_metrics.to_html(index=False, escape=True, classes="report-table")
observed_gap_metrics_html = observed_gap_metrics.to_html(index=False, escape=True, classes="report-table")
observed_relationship_metric_html = observed_relationship_metric.to_html(
    index=False,
    escape=True,
    classes="report-table",
)
ml_baseline_summary_html = ml_baseline_summary.to_html(index=False, escape=True, classes="report-table")
prescriptive_recommendations_html = prescriptive_recommendations.to_html(
    index=False,
    escape=True,
    classes="report-table",
)
final_limitations_html = final_limitations.to_html(index=False, escape=True, classes="report-table")

## 16 สร้างรายงาน HTML สรุป

รายงานเป็น static HTML และอ้างอิง Sarabun จากไฟล์ `../../Sarabun-Regular.ttf` เมื่อเบราว์เซอร์อนุญาตให้โหลด font ในตำแหน่งสัมพัทธ์ หากโหลดไม่ได้จะใช้ system font เป็น fallback


In [16]:
final_report_html = f"""<!doctype html>
<html lang="th">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>LABAI Final Summary Report</title>
  <style>
    @font-face {{
      font-family: "Sarabun";
      src: url("../../Sarabun-Regular.ttf") format("truetype");
      font-display: swap;
    }}
    :root {{
      color: #1d2939;
      background: #f7f8fa;
      font-family: "Sarabun", system-ui, sans-serif;
    }}
    body, h1, h2, h3, p, li, table, th, td, .notice, .metric, .metric-label, .metric-value, footer {{
      font-family: "Sarabun", system-ui, sans-serif;
    }}
    body {{
      margin: 0;
      background: #f7f8fa;
      font-family: "Sarabun", system-ui, sans-serif;
      line-height: 1.65;
    }}
    main {{
      max-width: 1120px;
      margin: 0 auto;
      padding: 32px 20px 48px;
    }}
    header {{
      border-bottom: 3px solid #1976a1;
      padding-bottom: 18px;
      margin-bottom: 24px;
    }}
    h1, h2, h3 {{
      color: #12374b;
      font-family: "Sarabun", system-ui, sans-serif;
      letter-spacing: 0;
    }}
    h1 {{
      font-size: 30px;
      margin: 0 0 6px;
    }}
    h2 {{
      font-size: 22px;
      border-bottom: 1px solid #d0d7de;
      padding-bottom: 6px;
      margin-top: 34px;
    }}
    h3 {{
      font-size: 18px;
      margin-bottom: 4px;
    }}
    p, li {{
      font-size: 16px;
    }}
    .lead {{
      margin: 0;
      color: #475467;
    }}
    .font-note {{
      color: #667085;
      font-size: 14px;
      margin: 10px 0 0;
    }}
    .notice {{
      background: #eaf3f7;
      border-left: 4px solid #1976a1;
      padding: 12px 16px;
      margin: 16px 0;
    }}
    .metric-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(190px, 1fr));
      gap: 12px;
      margin: 16px 0;
    }}
    .metric {{
      background: #ffffff;
      border: 1px solid #d0d7de;
      border-radius: 6px;
      padding: 14px;
    }}
    .metric-label {{
      color: #475467;
      font-size: 14px;
    }}
    .metric-value {{
      color: #12374b;
      font-size: 25px;
      font-weight: 700;
      margin-top: 2px;
    }}
    .report-table {{
      width: 100%;
      border-collapse: collapse;
      margin: 12px 0 20px;
      background: #ffffff;
      font-size: 14px;
    }}
    .report-table th, .report-table td {{
      border: 1px solid #d0d7de;
      padding: 9px;
      text-align: left;
      vertical-align: top;
    }}
    .report-table th {{
      background: #eaf3f7;
      color: #12374b;
    }}
    .link-list {{
      padding-left: 20px;
    }}
    a {{
      color: #075985;
    }}
    footer {{
      border-top: 1px solid #d0d7de;
      color: #667085;
      font-size: 14px;
      margin-top: 36px;
      padding-top: 16px;
    }}
    @media (max-width: 700px) {{
      main {{ padding: 20px 14px 32px; }}
      h1 {{ font-size: 25px; }}
      .report-table {{ display: block; overflow-x: auto; }}
    }}
  </style>
</head>
<body>
  <main>
    <header>
      <h1>รายงานสรุปโครงการ LABAI</h1>
      <p class="lead">สรุปข้อมูลที่สังเกตได้, baseline ML เพื่อการเรียนรู้, ข้อเสนอเชิงแนวทาง และข้อจำกัดของโครงการ</p>
      <p class="font-note">รายงานใช้ Sarabun จาก `../../Sarabun-Regular.ttf` เมื่อเบราว์เซอร์เข้าถึงไฟล์ได้ หากเข้าถึงไม่ได้ รายงานจะใช้ system font เป็น fallback และยังคงอ่านเนื้อหาได้</p>
    </header>

    <section>
      <h2>ภาพรวมโครงการ</h2>
      <p>โครงการนี้ใช้ข้อมูลสาธารณะของสถาบันบริหารจัดการธนาคารที่ดินเพื่อสาธิต workflow ตั้งแต่ค้นหาชุดข้อมูล ตรวจ schema ทำความสะอาด วิเคราะห์เชิงพรรณนาและเชิงวินิจฉัย ไปจนถึง baseline ML และข้อเสนอสำหรับการเก็บข้อมูลต่อไป</p>
      <div class="notice">ผลของข้อมูลและ model ในรายงานนี้มีไว้เพื่อการเรียนรู้และตรวจประเด็นข้อมูลต่อ ไม่ใช้จัดอันดับ ตัดสิน หรือกำหนดการดำเนินงานจริงของจังหวัด</div>
    </section>

    <section>
      <h2>ลำดับการทำงานจากข้อมูลดิบสู่ข้อเสนอ</h2>
      {workflow_steps_html}
    </section>

    <section>
      <h2>แหล่งข้อมูลที่ใช้</h2>
      {dataset_sources_html}
    </section>

    <section>
      <h2>ผลที่สังเกตได้จาก cleaned data</h2>
      <p>ตัวเลขต่อไปนี้มาจากไฟล์ `labai_demand_supply_cleaned.csv` จำนวน {row_count} แถวที่ดาวน์โหลดในรอบโครงการนี้</p>
      <div class="metric-grid">
        <div class="metric"><div class="metric-label">ผลรวมผู้ประสงค์ใช้ที่ดิน</div><div class="metric-value">{total_want_land:,}</div></div>
        <div class="metric"><div class="metric-label">ผลรวมเจ้าของที่ดิน</div><div class="metric-value">{total_owner_land:,}</div></div>
        <div class="metric"><div class="metric-label">ส่วนต่างมากกว่าศูนย์</div><div class="metric-value">{positive_gap_count}</div></div>
        <div class="metric"><div class="metric-label">Pearson correlation</div><div class="metric-value">{demand_supply_correlation:.3f}</div></div>
      </div>
      {observed_total_metrics_html}
      {observed_gap_metrics_html}
      {observed_relationship_metric_html}
      <p>ส่วนต่างเป็นผลคำนวณจากสอง column และ correlation เป็นความสัมพันธ์เชิงเส้นของไฟล์นี้เท่านั้น จึงไม่ใช้บอกสาเหตุ คุณภาพ หรือความเร่งด่วนเชิงนโยบาย</p>
    </section>

    <section>
      <h2>สรุปการทดลอง ML เบื้องต้น</h2>
      <p>ตารางนี้อ้างอิงผลการทดลองที่บันทึกไว้แล้วจากการรันโมเดล baseline โดยไม่มีการฝึก model หรือคำนวณ metric ซ้ำในรายงานสรุปนี้</p>
      {ml_baseline_summary_html}
    </section>

    <section>
      <h2>ข้อเสนอจากผลวิเคราะห์</h2>
      <p>ข้อเสนอเน้นการยืนยัน metadata และเพิ่มความพร้อมของข้อมูลก่อนขยายการวิเคราะห์หรือประยุกต์ใช้ model</p>
      {prescriptive_recommendations_html}
    </section>

    <section>
      <h2>ข้อจำกัดที่ต้องคงไว้ในการตีความ</h2>
      {final_limitations_html}
    </section>

    <section>
      <h2>แนวคิดต่อยอดจากชุดข้อมูลอื่น</h2>
      <p>แนวคิดต่อไปนี้เป็นตัวเลือกจาก `docs/project-idea-gallery.md` ยังไม่ใช่งานที่ดำเนินการในรายงาน LABAI ปัจจุบัน</p>
      <ul>
        <li><strong>LABAI:</strong> เมื่อยืนยันสถานะข้อมูลการจับคู่แล้ว สามารถทบทวน funnel และสำรวจความต้องการกับจำนวนเจ้าของที่ดินตามช่วงเวลาได้</li>
        <li><strong>SACIT:</strong> ชุดข้อมูลการส่งออกและชุมชนหัตถกรรมอาจรองรับ trend, map และ forecasting baseline เมื่อมีช่วงเวลาและ schema จริงที่เพียงพอ</li>
        <li><strong>KPI:</strong> ข้อมูล governance, conflict และ peace อาจใช้ทำ dashboard หรือ map ได้หลังตรวจ resource, เงื่อนไขการเข้าถึง และนิยามตัวชี้วัด</li>
      </ul>
    </section>

    <section>
      <h2>รายงานและเอกสารสำหรับทบทวน</h2>
      <ul class="link-list">
        <li><a href="interactive_data_report.html">รายงาน interactive แบบ offline</a></li>
        <li><a href="ml_baseline_report.html">รายงานการทดลอง ML เบื้องต้น</a></li>
        <li>สรุปโครงการ: อ่านได้ใน README.md ของ github_upload_clean</li>
        <li>ข้อเสนอเชิงแนวทาง: ดูได้ใน notebook 06 ส่วน prescriptive summary</li>
        <li>วิธีรัน notebook: ดูที่ docs/how_to_run.md</li>
      </ul>
    </section>

    <footer>สร้างจาก cleaned output และเอกสารประกอบโครงการ, วันที่อ้างอิงไฟล์ที่ดาวน์โหลด: 2026-07-10</footer>
  </main>
</body>
</html>
"""

## 17 บันทึกรายงาน HTML

สร้างโฟลเดอร์ปลายทางหากยังไม่มี และเขียนรายงานด้วย UTF-8 เพื่อรองรับข้อความภาษาไทย


In [17]:
final_report_path.parent.mkdir(parents=True, exist_ok=True)
final_report_path.write_text(final_report_html, encoding="utf-8")


13608

## 18 ตรวจสอบรายงานที่สร้าง

ตรวจ marker สำคัญของ report และยืนยันว่าไม่มีการอ้างอิง CDN ภายนอก รายงานนี้เป็น static HTML จึงไม่ต้องโหลด JavaScript เพื่อแสดงเนื้อหาหลัก


In [18]:
generated_report_html = final_report_path.read_text(encoding="utf-8")
required_report_markers = [
    "รายงานสรุปโครงการ LABAI",
    "ภาพรวมโครงการ",
    "ลำดับการทำงานจากข้อมูลดิบสู่ข้อเสนอ",
    "ผลที่สังเกตได้จาก cleaned data",
    "สรุปการทดลอง ML เบื้องต้น",
    "ข้อเสนอจากผลวิเคราะห์",
    "ข้อจำกัดที่ต้องคงไว้ในการตีความ",
    "แนวคิดต่อยอดจากชุดข้อมูลอื่น",
]

final_validation_results = pd.DataFrame(
    {
        "รายการตรวจ": [
            "มี section หลักครบ",
            "มี Sarabun font family",
            "มีข้อความ fallback ของ font",
            "มีลิงก์รายงาน interactive",
            "มีลิงก์รายงานการทดลอง ML",
            "ไม่มีการอ้างอิง CDN ภายนอก",
        ],
        "ผล": [
            all(marker in generated_report_html for marker in required_report_markers),
            'font-family: "Sarabun", system-ui, sans-serif' in generated_report_html,
            "system font เป็น fallback" in generated_report_html,
            "interactive_data_report.html" in generated_report_html,
            "ml_baseline_report.html" in generated_report_html,
            "cdn." not in generated_report_html,
        ],
    }
)

if not final_validation_results["ผล"].all():
    raise ValueError("รายงาน HTML ไม่ผ่านการตรวจ marker ที่กำหนด")

final_validation_results


,รายการตรวจ,ผล
0,มี section หลักครบ,True
1,มี Sarabun font family,True
2,มีข้อความ fallback ของ font,True
3,มีลิงก์รายงาน interactive,True
4,มีลิงก์รายงานการทดลอง ML,True
5,ไม่มีการอ้างอิง CDN ภายนอก,True
